# Resolution Comparison Experiment on COVIDx Dataset

This notebook explores the performance impact of image resolution on a Deep Learning model (ResNet18). We compare:
1. **Original Resolution Images**
2. **256x256 Resized Images**

This version uses the modular functions from the `functions` directory.

## 1. Setup and Configuration

In this section, we import the necessary libraries, configure the device, and add the `functions` path to the system path.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt

# Add functions directory to path
sys.path.append(os.path.abspath('functions'))

from train import train
from evaluation import plot_results, eval_on_metrics
from dataset import COVIDxResolutionDataset

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Paths
ROOT_DIR = '/home/furkan/Documents'
CSV_PATH = '/home/furkan/Documents/covidx/covidx_merged.csv'

## 2. Scenario 1: Original Resolution Training

Training on original images using the imported `train` function.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds_orig = COVIDxResolutionDataset(CSV_PATH, ROOT_DIR, split='train', transform=transform, use_resized=False)
val_ds_orig = COVIDxResolutionDataset(CSV_PATH, ROOT_DIR, split='val', transform=transform, use_resized=False)
test_ds_orig = COVIDxResolutionDataset(CSV_PATH, ROOT_DIR, split='test', transform=transform, use_resized=False)

train_loader_orig = DataLoader(train_ds_orig, batch_size=32, shuffle=True, num_workers=4)
val_loader_orig = DataLoader(val_ds_orig, batch_size=32, shuffle=False, num_workers=4)
test_loader_orig = DataLoader(test_ds_orig, batch_size=32, shuffle=False, num_workers=4)

model_orig = models.resnet18(weights='DEFAULT')
model_orig.fc = nn.Linear(model_orig.fc.in_features, 2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_orig.parameters(), lr=0.0001)

print("Training Original Resolution Model...")
logs_orig = train(
    model=model_orig, 
    train_loader=train_loader_orig, 
    val_loader=val_loader_orig, 
    criterion=criterion, 
    optimizer=optimizer, 
    device=device, 
    save_path='best_model_orig.pth', 
    num_epochs=5, 
    patience=3, 
    log_path='training_orig.log'
)

print("\nEvaluation on Test Data (Original):")
eval_on_metrics(model_orig, test_loader_orig)

## 3. Scenario 2: 256x256 Resized Training

Training on resized images using the same imported `train` function.

In [ ]:
train_ds_res = COVIDxResolutionDataset(CSV_PATH, ROOT_DIR, split='train', transform=transform, use_resized=True)
val_ds_res = COVIDxResolutionDataset(CSV_PATH, ROOT_DIR, split='val', transform=transform, use_resized=True)
test_ds_res = COVIDxResolutionDataset(CSV_PATH, ROOT_DIR, split='test', transform=transform, use_resized=True)

train_loader_res = DataLoader(train_ds_res, batch_size=32, shuffle=True, num_workers=4)
val_loader_res = DataLoader(val_ds_res, batch_size=32, shuffle=False, num_workers=4)
test_loader_res = DataLoader(test_ds_res, batch_size=32, shuffle=False, num_workers=4)

model_res = models.resnet18(weights='DEFAULT')
model_res.fc = nn.Linear(model_res.fc.in_features, 2)

optimizer_res = optim.Adam(model_res.parameters(), lr=0.0001)

print("Training 256x256 Resized Model...")
logs_res = train(
    model=model_res, 
    train_loader=train_loader_res, 
    val_loader=val_loader_res, 
    criterion=criterion, 
    optimizer=optimizer_res, 
    device=device, 
    save_path='best_model_res.pth', 
    num_epochs=5, 
    patience=3, 
    log_path='training_res.log'
)

print("\nEvaluation on Test Data (Resized):")
eval_on_metrics(model_res, test_loader_res)

## 4. Comparison of Results

Visualizing the training history using `plot_results`.

In [ ]:
print("Original Resolution curves:")
plot_results(*logs_orig)

print("\n256x256 Resolution curves:")
plot_results(*logs_res)